In [1]:
# ============================================================
# idiovol (GHZ 2016 SAS definition)
# - Build weekly returns from CRSP dsf (week ending SAT)
# - Equal-weight market weekly return = mean(wkret) across stocks each week
# - For each permno-month (calendar month-end label):
#     regress wkret on ewmkt over past 36 months excluding current month
#     require at least 52 weekly obs
#     idiovol = sample std of residuals
#
# Output: permno, date (calendar month-end), idiovol, n_weeks, alpha, beta
# Time: 1956-01 .. 1989-12
# ============================================================

import numpy as np
import pandas as pd
import wrds

# -----------------------
# 0) Parameters
# -----------------------
OUT_START = pd.Timestamp("1956-01-31")
OUT_END   = pd.Timestamp("1989-12-31")

# Need 36-month lookback; start pulling earlier for weekly construction
PULL_START_DAILY = pd.Timestamp("1953-01-01")
PULL_END_DAILY   = OUT_END

MIN_WEEKS = 52

# Output files
OUT_CSV = "idiovol_ghz_weekly_1956_1989.csv"
OUT_PARQUET = "idiovol_ghz_weekly_1956_1989.parquet"

# -----------------------
# 1) Connect WRDS
# -----------------------
db = wrds.Connection()

# -----------------------
# 2) Build the monthly "universe" of (permno, month_end) we want idiovol for
#    Use CRSP msf + msenames to filter common shares & major exchanges (like typical GHZ screens)
#    (If你不想筛 exchcd，可以把那行删掉)
# -----------------------
q_msf = f"""
select
  a.permno, a.date,
  b.shrcd, b.exchcd
from crsp.msf as a
left join crsp.msenames as b
  on a.permno = b.permno
 and b.namedt <= a.date
 and a.date <= b.nameendt
where a.date >= '{OUT_START.date()}'
  and a.date <= '{OUT_END.date()}'
"""
msf = db.raw_sql(q_msf, date_cols=["date"])

# Common shares only
msf = msf[msf["shrcd"].isin([10, 11])].copy()
# Major exchanges only (NYSE/AMEX/NASDAQ) -- 可选
msf = msf[msf["exchcd"].isin([1, 2, 3])].copy()

# Calendar month-end label (GHZ says aligned in calendar month)
msf["date_m"] = msf["date"].dt.to_period("M").dt.to_timestamp("M")

universe = msf[["permno", "date_m"]].drop_duplicates().sort_values(["permno","date_m"]).reset_index(drop=True)

print("Universe rows:", len(universe), "| unique permno:", universe["permno"].nunique(),
      "| months:", universe["date_m"].nunique())

# -----------------------
# 3) Pull CRSP daily returns from dsf and compute weekly returns
#    Weekly date = week ending Saturday (to match SAS intnx('WEEK',date,0,'end'))
#    wkret = exp(sum(log(1+ret))) - 1 within week
# -----------------------
q_dsf = f"""
select permno, date, ret
from crsp.dsf
where date >= '{PULL_START_DAILY.date()}'
  and date <= '{PULL_END_DAILY.date()}'
"""
dsf = db.raw_sql(q_dsf, date_cols=["date"])

# Clean ret: ret can be 'C' or missing; coerce to numeric
dsf["ret"] = pd.to_numeric(dsf["ret"], errors="coerce")
dsf = dsf.dropna(subset=["ret"]).copy()

# Week ending Saturday
# Using pandas period weekly ending SAT: 'W-SAT'
dsf["wkdt"] = dsf["date"].dt.to_period("W-SAT").dt.end_time.dt.normalize()

# Compute weekly return per permno-week: exp(sum(log1p(ret))) - 1
# (only for valid ret where 1+ret > 0)
valid = (1.0 + dsf["ret"]) > 0
dsf = dsf[valid].copy()
dsf["log1p_ret"] = np.log1p(dsf["ret"])

wk = (
    dsf.groupby(["permno", "wkdt"], as_index=False)["log1p_ret"]
       .sum()
)
wk["wkret"] = np.expm1(wk["log1p_ret"])
wk = wk.drop(columns=["log1p_ret"])

# Equal-weight market weekly return: mean of wkret across permnos each week
ew = wk.groupby("wkdt", as_index=False)["wkret"].mean().rename(columns={"wkret":"ewmkt"})
wk = wk.merge(ew, on="wkdt", how="left")

# Attach month label of each week-end date (calendar month-end)
wk["wk_month_end"] = wk["wkdt"].dt.to_period("M").dt.to_timestamp("M")

wk = wk.sort_values(["permno","wkdt"]).reset_index(drop=True)

print("Weekly rows:", len(wk), "| weeks:", wk["wkdt"].nunique(), "| permno:", wk["permno"].nunique())

# -----------------------
# 4) Compute idiovol for each (permno, month_end) in universe
#    Window: weeks with wkdt between [month_end-36M, month_end-1M]
#    Regression: y=wkret, x=ewmkt with intercept
#    idiovol = sample std of residuals (ddof=1), but computed via SSE/(n-1) since mean(resid)=0
# -----------------------
# Prepare lookup: for each permno, arrays of wkdt, x, y and cumulative sums for fast window stats
wk_groups = {}
for permno, g in wk.groupby("permno", sort=False):
    g = g.sort_values("wkdt")
    d = g["wkdt"].to_numpy("datetime64[ns]")
    x = g["ewmkt"].to_numpy(dtype=float)
    y = g["wkret"].to_numpy(dtype=float)

    # cumulative sums
    cx  = np.cumsum(x)
    cy  = np.cumsum(y)
    cxx = np.cumsum(x*x)
    cxy = np.cumsum(x*y)
    cyy = np.cumsum(y*y)

    wk_groups[int(permno)] = (d, x, y, cx, cy, cxx, cxy, cyy)

def window_sums(carr, lo, hi):
    """sum over inclusive [lo, hi] using cumulative array"""
    if lo > hi:
        return np.nan
    if lo == 0:
        return carr[hi]
    return carr[hi] - carr[lo-1]

out_rows = []

# Process permno by permno (much faster than per-row apply)
for permno, sub in universe.groupby("permno", sort=False):
    permno_int = int(permno)
    if permno_int not in wk_groups:
        continue

    d, x, y, cx, cy, cxx, cxy, cyy = wk_groups[permno_int]

    # months we need for this permno
    months = sub["date_m"].to_numpy("datetime64[ns]")

    for m in months:
        m_ts = pd.Timestamp(m)

        # window: [m-36M, m-1M]
        start = (m_ts.to_period("M") - 36).to_timestamp("M")  # month-end 36 months back
        end   = (m_ts.to_period("M") - 1 ).to_timestamp("M")  # month-end 1 month back

        start64 = np.datetime64(start.to_datetime64())
        end64   = np.datetime64(end.to_datetime64())

        # indices in weekly array where wkdt within [start, end]
        lo = np.searchsorted(d, start64, side="left")
        hi = np.searchsorted(d, end64, side="right") - 1

        n = hi - lo + 1
        if n < MIN_WEEKS:
            continue

        # Sums
        sumx  = window_sums(cx,  lo, hi)
        sumy  = window_sums(cy,  lo, hi)
        sumxx = window_sums(cxx, lo, hi)
        sumxy = window_sums(cxy, lo, hi)
        sumyy = window_sums(cyy, lo, hi)

        # OLS with intercept
        denom = n * sumxx - (sumx * sumx)
        if denom == 0 or np.isnan(denom):
            continue

        beta  = (n * sumxy - sumx * sumy) / denom
        alpha = (sumy - beta * sumx) / n

        # SSE = sum (y - alpha - beta x)^2 using sums
        # SSE = sumyy - 2*alpha*sumy - 2*beta*sumxy + n*alpha^2 + 2*alpha*beta*sumx + beta^2*sumxx
        sse = (
            sumyy
            - 2.0 * alpha * sumy
            - 2.0 * beta  * sumxy
            + n * alpha * alpha
            + 2.0 * alpha * beta * sumx
            + beta * beta * sumxx
        )

        # sample std of residuals (SAS std() uses n-1)
        # with intercept, mean(resid)=0, so var = SSE/(n-1)
        if n <= 1 or sse < 0:
            continue

        idiovol = np.sqrt(sse / (n - 1))

        out_rows.append((permno_int, m_ts, idiovol, n, alpha, beta))

res = pd.DataFrame(out_rows, columns=["permno","date","idiovol","n_weeks","alpha","beta"])

print("Result rows:", len(res), "| unique permno:", res["permno"].nunique())
if len(res) > 0:
    print("min date:", res["date"].min(), "| max date:", res["date"].max())

# -----------------------
# 5) Save
# -----------------------
res.to_csv(OUT_CSV, index=False)
res.to_parquet(OUT_PARQUET, index=False)

print("Saved:", OUT_CSV, "and", OUT_PARQUET)


Enter your WRDS username [zhouzixian]: zixian_zhou
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  n


You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Universe rows: 1433940 | unique permno: 13583 | months: 408
Weekly rows: 6819008 | weeks: 1931 | permno: 15056
Result rows: 1266278 | unique permno: 12403
min date: 1956-01-31 00:00:00 | max date: 1989-12-31 00:00:00
Saved: idiovol_ghz_weekly_1956_1989.csv and idiovol_ghz_weekly_1956_1989.parquet
